In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.impute import SimpleImputer
import xgboost as xgb
import category_encoders as ce
from category_encoders import CatBoostEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from hyperopt import fmin, tpe, hp, Trials
from ray import tune
# from ray.tune.suggest.hyperopt import HyperOptSearch ------> deprecated
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler
import warnings
warnings.filterwarnings('ignore')

/opt/anaconda3/envs/ml_final/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-01 21:14:19,503	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2024-12-01 21:14:19,734	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
!pwd

/Users/fffuuuming/Desktop/study/碩一上/HTML_2024_FAll/final_project/stage_1


# Load data

In [3]:
X = pd.read_csv('./combined_X.csv', low_memory=False)
y = pd.read_csv('./combined_y.csv', low_memory=False)

# Data processing : 

In [41]:
# Split the data into training (2016-2021) and testing (2022-2023)
train_df = df[df['season'].between(2016, 2021)].copy()
val_df = df[df['season'].between(2022, 2023)].copy()

# Check if the datasets are not empty
if train_df.empty or val_df.empty:
    raise ValueError("Training or testing data is empty. Please check the years in your dataset.")

# # Drop the 'date' and 'year' columns as they are not used in training
# train_df = train_df.drop(['date'], axis=1)
# test_df = test_df.drop(['date'], axis=1)

In [42]:
# Separate features and target variable
X_train = train_df.drop('home_team_win', axis=1)
y_train = train_df['home_team_win']

X_val = val_df.drop('home_team_win', axis=1)
y_val = val_df['home_team_win']

## Impute

In [43]:
# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Identify categorical columns with missing values
cat_cols_with_missing = X_train[categorical_cols].columns[
    X_train[categorical_cols].isnull().any()
].tolist()

# Identify numerical columns with missing values
num_cols_with_missing = X_train.select_dtypes(include=[np.number]).columns[
    X_train.select_dtypes(include=[np.number]).isnull().any()
].tolist()

In [44]:
# impute the training & test data on numerical columns
median_imputer = SimpleImputer(strategy='median')

X_train[num_cols_with_missing] = median_imputer.fit_transform(X_train[num_cols_with_missing])
X_val[num_cols_with_missing] = median_imputer.transform(X_val[num_cols_with_missing])

In [45]:
# impute the training & test data on category columns
mode_imputer = SimpleImputer(strategy='most_frequent')

X_train[cat_cols_with_missing] = mode_imputer.fit_transform(X_train[cat_cols_with_missing])
X_val[cat_cols_with_missing] = mode_imputer.transform(X_val[cat_cols_with_missing])

# Encoding

In [46]:
# Target Encoding
target_encoder = ce.TargetEncoder(cols=categorical_cols, smoothing=1)

X_train_target_encoded = X_train.copy()
X_train_target_encoded[categorical_cols] = target_encoder.fit_transform(X_train_target_encoded[categorical_cols], y_train)

X_val_target_encoded = X_val.copy()
X_val_target_encoded[categorical_cols] = target_encoder.transform(X_val_target_encoded[categorical_cols])

In [47]:
# CatBoost Encoding
catboost_encoder = CatBoostEncoder(cols=X_train.select_dtypes(include='object').columns)

X_train_catboost_encoded = X_train.copy()
X_train_catboost_encoded = catboost_encoder.fit_transform(X_train_catboost_encoded, y_train)

X_val_catboost_encoded = X_val.copy()
X_val_catboost_encoded = catboost_encoder.transform(X_val_catboost_encoded)

# Baseline Model Evaluation

In [48]:
baseline_model = xgb.XGBClassifier(objective='binary:logistic', random_state=13, n_jobs=-1)
baseline_model.fit(X_train_catboost_encoded, y_train)
baseline_preds = baseline_model.predict(X_val_catboost_encoded)
baseline_proba = baseline_model.predict_proba(X_val_catboost_encoded)[:, 1]

baseline_accuracy = accuracy_score(y_val, baseline_preds)
baseline_roc_auc = roc_auc_score(y_val, baseline_proba)

print(f"Baseline Model:\n- Accuracy: {baseline_accuracy * 100:.2f}%\n- ROC AUC: {baseline_roc_auc:.3f}")

Baseline Model:
- Accuracy: 53.77%
- ROC AUC: 0.535


# Optimization :

In [27]:


# Step 4: Hyperparameter Optimization with Ray Tune and Hyperopt
def train_xgb(config):
    model = xgb.XGBClassifier(
        **config,
        objective='binary:logistic',
        random_state=13,
        n_jobs=-1
    )
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], early_stopping_rounds=15, verbose=False)
    preds = model.predict(X_valid)
    proba = model.predict_proba(X_valid)[:, 1]
    roc_auc = roc_auc_score(y_valid, proba)
    tune.report(roc_auc=roc_auc)

search_space = {
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'reg_alpha': hp.loguniform('reg_alpha', np.log(1e-3), np.log(1)),
    'gamma': hp.uniform('gamma', 0, 1)
}

analysis = tune.run(
    train_xgb,
    config=search_space,
    metric="roc_auc",
    mode="max",
    num_samples=50,
    scheduler=ASHAScheduler(metric="roc_auc", mode="max"),
    search_alg=HyperOptSearch()
)

best_params = analysis.best_config
best_params['max_depth'] = int(best_params['max_depth'])
print("Best hyperparameters found:", best_params)

2024-11-17 22:42:20,714	INFO worker.py:1819 -- Started a local Ray instance.
2024-11-17 22:42:21,396	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `tune.run(...)`.
2024-11-17 22:42:21,398	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


RuntimeError: Trying to sample a configuration from HyperOptSearch, but no search space has been defined. Either pass the `space` argument when instantiating the search algorithm, or pass a `param_space` to `tune.Tuner()`. This issue can also come up with HyperOpt if your search space only contains constant variables, which is not supported by HyperOpt. In that case, don't pass any searcher or add sample variables to the search space.

In [ ]:
# Step 5: Train Final Model with Optimized Parameters
final_model = xgb.XGBClassifier(**best_params, objective='binary:logistic', random_state=13, n_jobs=-1)
final_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], early_stopping_rounds=15, verbose=True)

In [ ]:
# Step 6: Feature Importance Plot
feature_importances = final_model.get_booster().get_score(importance_type='gain')
sorted_features = pd.Series(feature_importances).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
sorted_features[:30].plot(kind='barh', color='steelblue')
plt.title('Top 30 Feature Importances by Gain')
plt.show()

In [ ]:
# Step 7: Confidence Analysis and Prediction Probability Binning
valid_proba = final_model.predict_proba(X_valid)[:, 1]
confidence = np.abs(valid_proba - 0.5) + 0.5

In [ ]:
# Bin the prediction confidence
bins = np.linspace(0.5, 1.0, 10)
df_valid = pd.DataFrame({'proba': valid_proba, 'confidence': confidence, 'true_label': y_valid})
df_valid['probability_bin'] = pd.cut(df_valid['confidence'], bins)

plt.figure(figsize=(10, 6))
sns.barplot(x='probability_bin', y='true_label', data=df_valid, estimator=np.mean, palette="Blues_d")
plt.title('Actual Win Percentage vs. Model Prediction Confidence')
plt.xlabel('Prediction Probability Bin')
plt.ylabel('Actual Win Percentage')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Step 8: Performance Reporting
final_preds = final_model.predict(X_valid)
final_accuracy = accuracy_score(y_valid, final_preds)
final_roc_auc = roc_auc_score(y_valid, valid_proba)

print(f"Final Model Performance:\n- Accuracy: {final_accuracy * 100:.2f}%\n- ROC AUC: {final_roc_auc:.3f}")